In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

folder = '/content/drive/MyDrive/physician-analytics'
print(os.listdir(folder))

['Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2022.csv', 'tableau_exports', 'output']


In [4]:
# Install PySpark - takes about 30 seconds in Colab
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PhysicianBehavioralAnalytics") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Success!")

Spark version: 4.0.2
Success!


In [5]:
# Define file path
file_path = '/content/drive/MyDrive/physician-analytics/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2022.csv'

# Read with PySpark - inferSchema automatically detects column types
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(file_path)

# Basic overview
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print("\nSchema:")
df.printSchema()

Total rows: 9,755,427
Total columns: 28

Schema:
root
 |-- Rndrng_NPI: integer (nullable = true)
 |-- Rndrng_Prvdr_Last_Org_Name: string (nullable = true)
 |-- Rndrng_Prvdr_First_Name: string (nullable = true)
 |-- Rndrng_Prvdr_MI: string (nullable = true)
 |-- Rndrng_Prvdr_Crdntls: string (nullable = true)
 |-- Rndrng_Prvdr_Ent_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_St1: string (nullable = true)
 |-- Rndrng_Prvdr_St2: string (nullable = true)
 |-- Rndrng_Prvdr_City: string (nullable = true)
 |-- Rndrng_Prvdr_State_Abrvtn: string (nullable = true)
 |-- Rndrng_Prvdr_State_FIPS: string (nullable = true)
 |-- Rndrng_Prvdr_Zip5: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA: double (nullable = true)
 |-- Rndrng_Prvdr_RUCA_Desc: string (nullable = true)
 |-- Rndrng_Prvdr_Cntry: string (nullable = true)
 |-- Rndrng_Prvdr_Type: string (nullable = true)
 |-- Rndrng_Prvdr_Mdcr_Prtcptg_Ind: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullab

In [6]:
from pyspark.sql import functions as F

# Select relevant columns and rename for clarity
df_clean = df.select(
    F.col("Rndrng_NPI").alias("npi"),
    F.col("Rndrng_Prvdr_Last_Org_Name").alias("last_name"),
    F.col("Rndrng_Prvdr_First_Name").alias("first_name"),
    F.col("Rndrng_Prvdr_Crdntls").alias("credentials"),
    F.col("Rndrng_Prvdr_Type").alias("specialty"),
    F.col("Rndrng_Prvdr_State_Abrvtn").alias("state"),
    F.col("Rndrng_Prvdr_City").alias("city"),
    F.col("Rndrng_Prvdr_Zip5").alias("zip"),
    F.col("Rndrng_Prvdr_Mdcr_Prtcptg_Ind").alias("medicare_participating"),
    F.col("HCPCS_Cd").alias("hcpcs_code"),
    F.col("HCPCS_Desc").alias("hcpcs_desc"),
    F.col("HCPCS_Drug_Ind").alias("is_drug"),
    F.col("Place_Of_Srvc").alias("place_of_service"),
    F.col("Tot_Benes").alias("total_beneficiaries"),
    F.col("Tot_Srvcs").alias("total_services"),
    F.col("Tot_Bene_Day_Srvcs").alias("total_bene_days"),
    F.col("Avg_Sbmtd_Chrg").alias("avg_submitted_charge"),
    F.col("Avg_Mdcr_Alowd_Amt").alias("avg_medicare_allowed"),
    F.col("Avg_Mdcr_Pymt_Amt").alias("avg_medicare_payment"),
    F.col("Avg_Mdcr_Stdzd_Amt").alias("avg_standardized_payment")
)

print(f"Columns kept: {len(df_clean.columns)}")
print(f"Columns dropped: {28 - len(df_clean.columns)}")
df_clean.printSchema()

Columns kept: 20
Columns dropped: 8
root
 |-- npi: integer (nullable = true)
 |-- last_name: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- credentials: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- medicare_participating: string (nullable = true)
 |-- hcpcs_code: string (nullable = true)
 |-- hcpcs_desc: string (nullable = true)
 |-- is_drug: string (nullable = true)
 |-- place_of_service: string (nullable = true)
 |-- total_beneficiaries: integer (nullable = true)
 |-- total_services: double (nullable = true)
 |-- total_bene_days: integer (nullable = true)
 |-- avg_submitted_charge: double (nullable = true)
 |-- avg_medicare_allowed: double (nullable = true)
 |-- avg_medicare_payment: double (nullable = true)
 |-- avg_standardized_payment: double (nullable = true)



In [7]:
# Fast missing value check - single pass through data
print("=== MISSING VALUES ===")
null_counts = df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).collect()[0].asDict()

total = df_clean.count()
for col_name, null_count in null_counts.items():
    if null_count > 0:
        pct = (null_count / total) * 100
        print(f"{col_name}: {null_count:,} nulls ({pct:.1f}%)")

print("\n=== SAMPLE DATA ===")
df_clean.show(5, truncate=False)

print("\n=== KEY STATS ===")
df_clean.select(
    "total_beneficiaries",
    "total_services",
    "avg_submitted_charge",
    "avg_medicare_payment"
).describe().show()

=== MISSING VALUES ===
first_name: 545,285 nulls (5.6%)
credentials: 1,049,271 nulls (10.8%)

=== SAMPLE DATA ===
+----------+---------+----------+-----------+-----------------+-----+--------+-----+----------------------+----------+---------------------------------------------------------------+-------+----------------+-------------------+--------------+---------------+--------------------+--------------------+--------------------+------------------------+
|npi       |last_name|first_name|credentials|specialty        |state|city    |zip  |medicare_participating|hcpcs_code|hcpcs_desc                                                     |is_drug|place_of_service|total_beneficiaries|total_services|total_bene_days|avg_submitted_charge|avg_medicare_allowed|avg_medicare_payment|avg_standardized_payment|
+----------+---------+----------+-----------+-----------------+-----+--------+-----+----------------------+----------+---------------------------------------------------------------+-------+--

In [8]:
print("=== UNIQUE PROVIDERS ===")
print(f"Unique NPIs: {df_clean.select('npi').distinct().count():,}")

print("\n=== TOP 10 STATES BY PROVIDER COUNT ===")
df_clean.groupBy("state") \
    .agg(F.countDistinct("npi").alias("provider_count")) \
    .orderBy(F.desc("provider_count")) \
    .show(10)

print("\n=== TOP 10 SPECIALTIES ===")
df_clean.groupBy("specialty") \
    .agg(F.countDistinct("npi").alias("provider_count")) \
    .orderBy(F.desc("provider_count")) \
    .show(10, truncate=False)

=== UNIQUE PROVIDERS ===
Unique NPIs: 1,148,873

=== TOP 10 STATES BY PROVIDER COUNT ===
+-----+--------------+
|state|provider_count|
+-----+--------------+
|   CA|         90432|
|   NY|         79417|
|   TX|         75890|
|   FL|         74374|
|   PA|         57253|
|   OH|         46854|
|   IL|         46243|
|   MI|         39395|
|   NC|         39219|
|   NJ|         35339|
+-----+--------------+
only showing top 10 rows

=== TOP 10 SPECIALTIES ===
+---------------------------------------------+--------------+
|specialty                                    |provider_count|
+---------------------------------------------+--------------+
|Nurse Practitioner                           |162859        |
|Physician Assistant                          |88954         |
|Internal Medicine                            |88132         |
|Family Practice                              |78867         |
|Physical Therapist in Private Practice       |69322         |
|Emergency Medicine             

In [9]:
# Filter to top 5 states and cache for faster queries
top_5_states = ['CA', 'NY', 'TX', 'FL', 'PA']

df_filtered = df_clean.filter(F.col("state").isin(top_5_states))

# Cache in memory — makes all future queries much faster
df_filtered.cache()

# Trigger the cache
total_filtered = df_filtered.count()
print(f"Total rows after filtering to top 5 states: {total_filtered:,}")

# Provider count per state
print("\n=== PROVIDERS PER STATE ===")
df_filtered.groupBy("state") \
    .agg(F.countDistinct("npi").alias("unique_providers"),
         F.count("*").alias("total_records")) \
    .orderBy(F.desc("unique_providers")) \
    .show()

Total rows after filtering to top 5 states: 3,244,948

=== PROVIDERS PER STATE ===
+-----+----------------+-------------+
|state|unique_providers|total_records|
+-----+----------------+-------------+
|   CA|           90432|       797629|
|   NY|           79417|       622553|
|   TX|           75890|       675859|
|   FL|           74374|       706472|
|   PA|           57253|       442435|
+-----+----------------+-------------+



In [10]:
# ANALYSIS 1 — Top specialties by utilization and payment
specialty_analysis = df_filtered.groupBy("specialty") \
    .agg(
        F.countDistinct("npi").alias("unique_providers"),
        F.sum("total_services").alias("total_services"),
        F.sum("total_beneficiaries").alias("total_beneficiaries"),
        F.avg("avg_medicare_payment").alias("avg_medicare_payment"),
        F.avg("avg_submitted_charge").alias("avg_submitted_charge")
    ) \
    .withColumn("payment_ratio",
        F.round(F.col("avg_medicare_payment") / F.col("avg_submitted_charge") * 100, 1)
    ) \
    .orderBy(F.desc("total_services")) \
    .limit(20)

specialty_analysis.show(20, truncate=False)

+--------------------------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------+
|specialty                             |unique_providers|total_services      |total_beneficiaries|avg_medicare_payment|avg_submitted_charge|payment_ratio|
+--------------------------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------+
|Clinical Laboratory                   |1056            |1.1885018300000003E8|58776256           |43.31328179013181   |131.277949287388    |33.0         |
|Hematology-Oncology                   |3118            |9.82198744E7        |4323288            |55.83201944830854   |229.09862394181528  |24.4         |
|Diagnostic Radiology                  |11221           |7.68482915E7        |25850032           |55.198019333728666  |362.99821342954175  |15.2         |
|Internal Medicine                     |32814           |5.37943465E7 

In [11]:
# ANALYSIS 2 — State level payment efficiency
state_analysis = df_filtered.groupBy("state") \
    .agg(
        F.countDistinct("npi").alias("unique_providers"),
        F.sum("total_services").alias("total_services"),
        F.sum("total_beneficiaries").alias("total_beneficiaries"),
        F.avg("avg_medicare_payment").alias("avg_medicare_payment"),
        F.avg("avg_submitted_charge").alias("avg_submitted_charge"),
        F.sum(F.col("avg_medicare_payment") * F.col("total_services")).alias("total_medicare_paid")
    ) \
    .withColumn("payment_ratio",
        F.round(F.col("avg_medicare_payment") / F.col("avg_submitted_charge") * 100, 1)
    ) \
    .orderBy(F.desc("total_medicare_paid"))

state_analysis.show(truncate=False)

+-----+----------------+--------------------+-------------------+--------------------+--------------------+---------------------+-------------+
|state|unique_providers|total_services      |total_beneficiaries|avg_medicare_payment|avg_submitted_charge|total_medicare_paid  |payment_ratio|
+-----+----------------+--------------------+-------------------+--------------------+--------------------+---------------------+-------------+
|CA   |90432           |2.551331753E8       |79260093           |97.35734567063862   |459.4454417778783   |1.0675572538823685E10|21.2         |
|FL   |74374           |2.731503848000001E8 |73089283           |94.5382418140087    |452.6515869918165   |8.324954008710107E9  |20.9         |
|TX   |75890           |1.8769518920000005E8|59885683           |86.27349355245121   |466.46175116260775  |6.472468682199377E9  |18.5         |
|NY   |79417           |1.5273006189999998E8|51320148           |87.02653729498545   |459.76429264400826  |6.238194738689777E9  |18.9   

In [12]:
# ANALYSIS 3 — Top procedures by volume and payment
procedure_analysis = df_filtered.groupBy("hcpcs_code", "hcpcs_desc", "is_drug") \
    .agg(
        F.sum("total_services").alias("total_services"),
        F.sum("total_beneficiaries").alias("total_beneficiaries"),
        F.avg("avg_medicare_payment").alias("avg_medicare_payment"),
        F.avg("avg_submitted_charge").alias("avg_submitted_charge"),
        F.countDistinct("npi").alias("unique_providers")
    ) \
    .withColumn("payment_ratio",
        F.round(F.col("avg_medicare_payment") / F.col("avg_submitted_charge") * 100, 1)
    ) \
    .orderBy(F.desc("total_services")) \
    .limit(25)

procedure_analysis.show(25, truncate=False)

+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+--------------+-------------------+--------------------+--------------------+----------------+-------------+
|hcpcs_code|hcpcs_desc                                                                                                                                                                           |is_drug|total_services|total_beneficiaries|avg_medicare_payment|avg_submitted_charge|unique_providers|payment_ratio|
+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+--------------+-------------------+--------------------+--------------------+----------------+-------------+
|Q9967     |Low osmolar contrast material, 300-399 mg/ml iodine con

In [13]:
# ANALYSIS 4 — Provider segments (high volume vs high value)
provider_segments = df_filtered.groupBy("npi", "last_name", "first_name", "specialty", "state") \
    .agg(
        F.sum("total_services").alias("total_services"),
        F.sum("total_beneficiaries").alias("total_beneficiaries"),
        F.avg("avg_medicare_payment").alias("avg_medicare_payment"),
        F.countDistinct("hcpcs_code").alias("unique_procedures")
    ) \
    .withColumn("segment",
        F.when(
            (F.col("total_services") > 1000) & (F.col("avg_medicare_payment") > 100),
            "High Volume High Value"
        ).when(
            F.col("total_services") > 1000,
            "High Volume Low Value"
        ).when(
            F.col("avg_medicare_payment") > 100,
            "Low Volume High Value"
        ).otherwise("Low Volume Low Value")
    )

# Segment summary
provider_segments.groupBy("segment") \
    .agg(
        F.count("*").alias("provider_count"),
        F.avg("total_services").alias("avg_services"),
        F.avg("avg_medicare_payment").alias("avg_payment")
    ) \
    .orderBy(F.desc("provider_count")) \
    .show(truncate=False)

+----------------------+--------------+-----------------+------------------+
|segment               |provider_count|avg_services     |avg_payment       |
+----------------------+--------------+-----------------+------------------+
|Low Volume Low Value  |195686        |314.6211282360516|60.883885075879355|
|High Volume Low Value |93028         |7509.368653523668|55.97200899083555 |
|Low Volume High Value |61990         |297.7417535086304|163.64485812893167|
|High Volume High Value|26662         |7180.488560498082|197.90443115448716|
+----------------------+--------------+-----------------+------------------+



In [14]:
import pandas as pd

output_folder = '/content/drive/MyDrive/physician-analytics/tableau_exports'
os.makedirs(output_folder, exist_ok=True)

# Convert to pandas and export
specialty_analysis.toPandas().to_csv(
    f'{output_folder}/tableau_specialties.csv', index=False)
print("✓ Specialties exported")

state_analysis.toPandas().to_csv(
    f'{output_folder}/tableau_states.csv', index=False)
print("✓ States exported")

procedure_analysis.toPandas().to_csv(
    f'{output_folder}/tableau_procedures.csv', index=False)
print("✓ Procedures exported")

provider_segments.groupBy("segment", "specialty", "state") \
    .agg(
        F.count("*").alias("provider_count"),
        F.avg("total_services").alias("avg_services"),
        F.avg("avg_medicare_payment").alias("avg_payment")
    ).toPandas().to_csv(
    f'{output_folder}/tableau_segments.csv', index=False)
print("✓ Segments exported")

print("\nAll files exported to Google Drive!")

✓ Specialties exported
✓ States exported
✓ Procedures exported
✓ Segments exported

All files exported to Google Drive!


In [15]:
# Heat map data — specialty x state breakdown
heatmap_data = df_filtered.groupBy("specialty", "state") \
    .agg(
        F.avg("avg_medicare_payment").alias("avg_medicare_payment"),
        F.sum("total_services").alias("total_services"),
        F.countDistinct("npi").alias("unique_providers")
    ) \
    .filter(F.col("unique_providers") >= 50)  # Remove sparse combos

# Limit to top 15 specialties for readability
top_specialties = df_filtered.groupBy("specialty") \
    .agg(F.sum("total_services").alias("total")) \
    .orderBy(F.desc("total")) \
    .limit(15) \
    .select("specialty")

heatmap_filtered = heatmap_data.join(top_specialties, on="specialty", how="inner")

heatmap_filtered.toPandas().to_csv(
    f'{output_folder}/tableau_heatmap.csv', index=False)
print(f"Heatmap rows: {heatmap_filtered.count()}")
print("✓ Heatmap exported")

Heatmap rows: 75
✓ Heatmap exported


In [16]:
# Treemap data — specialty → procedure hierarchy
treemap_data = df_filtered.groupBy("specialty", "hcpcs_desc") \
    .agg(
        F.sum("total_services").alias("total_services"),
        F.avg("avg_medicare_payment").alias("avg_medicare_payment"),
        F.countDistinct("npi").alias("unique_providers")
    ) \
    .filter(F.col("unique_providers") >= 100)

# Limit to top 10 specialties
top_10 = df_filtered.groupBy("specialty") \
    .agg(F.sum("total_services").alias("total")) \
    .orderBy(F.desc("total")) \
    .limit(10) \
    .select("specialty")

treemap_filtered = treemap_data.join(top_10, on="specialty", how="inner")

treemap_filtered.toPandas().to_csv(
    f'{output_folder}/tableau_treemap.csv', index=False)
print(f"Treemap rows: {treemap_filtered.count()}")
print("✓ Treemap exported")

Treemap rows: 1042
✓ Treemap exported
